In [0]:
import pandas as pd

df = pd.read_csv("retail_store_sales.csv")
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [0]:
print(df.shape)
df.isna().sum()

(12575, 11)


Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [0]:
df[df['Item'].isna() & df['Quantity'].isna()].shape

(604, 11)

In [0]:
df[df['Item'].isna() & df['Price Per Unit'].isna()].shape

(609, 11)

In [0]:
df[df['Total Spent'].isna() & df['Quantity'].isna()].shape

(604, 11)

In [0]:
df[df['Item'].isna() & df['Discount Applied'].isna()].shape

(416, 11)

In [0]:
df['Item'].unique()

array(['Item_10_PAT', 'Item_17_MILK', 'Item_12_BUT', 'Item_16_BEV',
       'Item_6_FOOD', nan, 'Item_1_FOOD', 'Item_16_FUR', 'Item_22_BUT',
       'Item_3_BUT', 'Item_2_FOOD', 'Item_24_PAT', 'Item_16_MILK',
       'Item_17_PAT', 'Item_13_EHE', 'Item_7_BEV', 'Item_4_EHE',
       'Item_10_FOOD', 'Item_14_FUR', 'Item_20_BUT', 'Item_25_FUR',
       'Item_14_FOOD', 'Item_22_PAT', 'Item_11_FOOD', 'Item_6_PAT',
       'Item_21_EHE', 'Item_25_BEV', 'Item_23_FOOD', 'Item_10_FUR',
       'Item_11_BEV', 'Item_23_BUT', 'Item_22_BEV', 'Item_10_EHE',
       'Item_24_BUT', 'Item_8_BEV', 'Item_3_FOOD', 'Item_12_FOOD',
       'Item_16_CEA', 'Item_11_PAT', 'Item_16_BUT', 'Item_5_CEA',
       'Item_19_MILK', 'Item_23_FUR', 'Item_7_FUR', 'Item_15_CEA',
       'Item_6_MILK', 'Item_24_CEA', 'Item_22_CEA', 'Item_22_FOOD',
       'Item_2_BUT', 'Item_14_PAT', 'Item_12_PAT', 'Item_18_FOOD',
       'Item_1_PAT', 'Item_4_BEV', 'Item_22_FUR', 'Item_7_PAT',
       'Item_20_CEA', 'Item_20_FOOD', 'Item_11_FUR', 'Item

In [0]:
df['Category'].unique()

array(['Patisserie', 'Milk Products', 'Butchers', 'Beverages', 'Food',
       'Furniture', 'Electric household essentials',
       'Computers and electric accessories'], dtype=object)

In [0]:
df['Payment Method'].unique()

array(['Digital Wallet', 'Credit Card', 'Cash'], dtype=object)

In [0]:
df['Location'].unique()

array(['Online', 'In-store'], dtype=object)

In [0]:
df['Discount Applied'].unique()

array([True, False, nan], dtype=object)

### Cleaning the Data

---

1. **Drop rows where Item is null (1,213 rows = ~9.6% of data)**

- You can't analyze sales or create meaningful dashboard visualizations without knowing what product was sold
- These records have no business value for reporting

2. **Drop rows where core sales metrics are null:**

- Quantity, Price Per Unit, or Total Spent missing → can't calculate revenue or analyze purchase behavior
- The overlap analysis shows 604 rows have multiple critical nulls together (Item + Quantity, Total Spent + Quantity)

3. **KEEP and FILL Discount Applied nulls (4,199 rows):**

- These likely represent transactions with no discount (legitimate business scenario)



In [0]:
# Drop rows with critical nulls
df_clean = df.dropna(subset=['Item', 'Quantity', 'Price Per Unit', 'Total Spent']).copy()

df_clean.isna().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                   0
Price Per Unit         0
Quantity               0
Total Spent            0
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    3783
dtype: int64

In [0]:
df_clean['Discount Applied'] = df_clean['Discount Applied'].fillna(False)

df_clean.isna().sum()

/home/spark-ad2bb946-0ffb-4999-99b7-b9/.ipykernel/12877/command-5543806833988675-1341196116:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean['Discount Applied'] = df_clean['Discount Applied'].fillna(False)


Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

In [0]:
df_clean.shape

(11362, 11)

### Feature Engineering

---

In [0]:
# Convert Transaction Date to datetime
df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'])

# Extract date components for time-based analysis
df_clean['Year'] = df_clean['Transaction Date'].dt.year
df_clean['Month'] = df_clean['Transaction Date'].dt.month
df_clean['Month_Name'] = df_clean['Transaction Date'].dt.strftime('%B')
df_clean['Quarter'] = df_clean['Transaction Date'].dt.quarter
df_clean['Day'] = df_clean['Transaction Date'].dt.day
df_clean['Day_of_Week'] = df_clean['Transaction Date'].dt.day_name()
df_clean['Week_of_Year'] = df_clean['Transaction Date'].dt.isocalendar().week

# Extract Item Number for product line analysis
df_clean['Item_Number'] = df_clean['Item'].str.extract(r'Item_(\d+)_')[0].astype(int)

print(f"Feature Engineering Complete!")
print(f"New columns added: {df_clean.shape[1] - 11} features")
df_clean.head()

Feature Engineering Complete!
New columns added: 8 features


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Year,Month,Month_Name,Quarter,Day,Day_of_Week,Week_of_Year,Item_Number
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4,April,2,8,Monday,15,10
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7,July,3,23,Sunday,29,17
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10,October,4,5,Wednesday,40,12
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,2022,5,May,2,7,Saturday,18,16
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10,October,4,2,Sunday,39,6


In [0]:
# Save the cleaned and engineered dataset
output_filename = 'retail_store_sales_cleaned.csv'
df_clean.to_csv(output_filename, index=False)

print(f"✓ Cleaned dataset saved to: {output_filename}")
print(f"\nColumn List ({df_clean.shape[1]} total):")
for i, col in enumerate(df_clean.columns, 1):
    print(f"{i:2d}. {col}")

✓ Cleaned dataset saved to: retail_store_sales_cleaned.csv

Column List (28 total):
 1. Transaction ID
 2. Customer ID
 3. Category
 4. Item
 5. Price Per Unit
 6. Quantity
 7. Total Spent
 8. Payment Method
 9. Location
10. Transaction Date
11. Discount Applied
12. Year
13. Month
14. Month_Name
15. Quarter
16. Day
17. Day_of_Week
18. Week_of_Year
19. Item_Number
20. Revenue_Without_Discount
21. Discount_Amount
22. Discount_Percentage
23. Revenue_Segment
24. Quantity_Segment
25. Transaction_Count
26. Total_Customer_Value
27. Recency_Days
28. Customer_Segment


In [0]:
# Calculate customer-level RFM (Recency, Frequency, Monetary) features
customer_stats = df_clean.groupby('Customer ID').agg({
    'Transaction Date': ['max', 'count'],
    'Total Spent': 'sum'
}).reset_index()

customer_stats.columns = ['Customer ID', 'Last_Transaction_Date', 'Transaction_Count', 'Total_Customer_Value']

# Calculate recency (days since last transaction) - using max date in dataset as reference
reference_date = df_clean['Transaction Date'].max()
customer_stats['Recency_Days'] = (reference_date - customer_stats['Last_Transaction_Date']).dt.days

# Merge back to main dataframe
df_clean = df_clean.merge(customer_stats[['Customer ID', 'Transaction_Count', 'Total_Customer_Value', 'Recency_Days']], 
                          on='Customer ID', 
                          how='left')

# Create customer segments based on Total Customer Value
df_clean['Customer_Segment'] = pd.cut(df_clean['Total_Customer_Value'],
                                       bins=[0, 1000, 5000, 10000, float('inf')],
                                       labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

print("Customer RFM Features Added!")
print(f"\nDataset Shape: {df_clean.shape}")
print(f"Total Features: {df_clean.shape[1]}")
df_clean[['Customer ID', 'Transaction_Count', 'Total_Customer_Value', 'Customer_Segment']].drop_duplicates('Customer ID').head(10)

Customer RFM Features Added!

Dataset Shape: (11362, 28)
Total Features: 28


,Customer ID,Transaction_Count,Total_Customer_Value,Customer_Segment
0,CUST_09,465,57113.0,Platinum
1,CUST_22,456,59460.0,Platinum
2,CUST_02,447,59512.5,Platinum
3,CUST_06,443,56020.0,Platinum
4,CUST_05,497,63855.5,Platinum
5,CUST_07,433,57084.5,Platinum
6,CUST_23,450,59738.5,Platinum
7,CUST_25,438,55732.5,Platinum
10,CUST_14,447,58076.5,Platinum
13,CUST_17,422,53790.5,Platinum


In [0]:
# Calculate discount metrics
# Assume Total Spent is after discount, calculate original price
df_clean['Revenue_Without_Discount'] = df_clean['Price Per Unit'] * df_clean['Quantity']
df_clean['Discount_Amount'] = df_clean['Revenue_Without_Discount'] - df_clean['Total Spent']
df_clean['Discount_Percentage'] = (df_clean['Discount_Amount'] / df_clean['Revenue_Without_Discount'] * 100).round(2)

# Handle cases where discount wasn't applied (set to 0)
df_clean.loc[df_clean['Discount Applied'] == False, 'Discount_Amount'] = 0
df_clean.loc[df_clean['Discount Applied'] == False, 'Discount_Percentage'] = 0

# Create revenue segments
df_clean['Revenue_Segment'] = pd.cut(df_clean['Total Spent'], 
                                      bins=[0, 100, 500, 1000, float('inf')],
                                      labels=['Low (<$100)', 'Medium ($100-$500)', 'High ($500-$1000)', 'Premium (>$1000)'])

# Create quantity segments
df_clean['Quantity_Segment'] = pd.cut(df_clean['Quantity'],
                                       bins=[0, 1, 3, 5, float('inf')],
                                       labels=['Single', 'Small Batch (2-3)', 'Medium Batch (4-5)', 'Large Batch (>5)'])

print("Revenue and Discount Metrics Added!")
df_clean[['Total Spent', 'Revenue_Without_Discount', 'Discount_Amount', 'Discount_Percentage', 'Revenue_Segment']].head()

Revenue and Discount Metrics Added!


,Total Spent,Revenue_Without_Discount,Discount_Amount,Discount_Percentage,Revenue_Segment
0,185.0,185.0,0.0,0.0,Medium ($100-$500)
1,261.0,261.0,0.0,0.0,Medium ($100-$500)
2,43.0,43.0,0.0,0.0,Low (<$100)
3,247.5,247.5,0.0,0.0,Medium ($100-$500)
4,87.5,87.5,0.0,0.0,Low (<$100)


In [0]:
df_clean.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Year,Month,Month_Name,Quarter,Day,Day_of_Week,Week_of_Year,Item_Number,Revenue_Without_Discount,Discount_Amount,Discount_Percentage,Revenue_Segment,Quantity_Segment,Transaction_Count,Total_Customer_Value,Recency_Days,Customer_Segment
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,2024,4,April,2,8,Monday,15,10,185.0,0.0,0.0,Medium ($100-$500),Large Batch (>5),465,57113.0,1,Platinum
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,2023,7,July,3,23,Sunday,29,17,261.0,0.0,0.0,Medium ($100-$500),Large Batch (>5),456,59460.0,3,Platinum
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,2022,10,October,4,5,Wednesday,40,12,43.0,0.0,0.0,Low (<$100),Small Batch (2-3),447,59512.5,2,Platinum
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,2022,5,May,2,7,Saturday,18,16,247.5,0.0,0.0,Medium ($100-$500),Large Batch (>5),443,56020.0,2,Platinum
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,2022,10,October,4,2,Sunday,39,6,87.5,0.0,0.0,Low (<$100),Large Batch (>5),497,63855.5,4,Platinum


In [0]:
df_clean.to_csv("final_data.csv")

In [0]:
# Final data quality checks
print("=" * 60)
print("DATA CLEANING & FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"\n✓ Original Dataset: 12,575 rows")
print(f"✓ Cleaned Dataset: {df_clean.shape[0]:,} rows ({df_clean.shape[0]/12575*100:.1f}% retained)")
print(f"✓ Total Features: {df_clean.shape[1]}")
print(f"✓ Missing Values: {df_clean.isna().sum().sum()}")

print("\n" + "=" * 60)
print("FEATURE CATEGORIES")
print("=" * 60)
print("\n1. Date Features:")
print("   - Year, Month, Month_Name, Quarter, Day, Day_of_Week, Week_of_Year")
print("\n2. Product Features:")
print("   - Item_Number (extracted from Item column)")
print("\n3. Revenue & Discount Features:")
print("   - Revenue_Without_Discount, Discount_Amount, Discount_Percentage")
print("   - Revenue_Segment, Quantity_Segment")
print("\n4. Customer RFM Features:")
print("   - Transaction_Count, Total_Customer_Value, Recency_Days")
print("   - Customer_Segment (Bronze/Silver/Gold/Platinum)")

print("\n" + "=" * 60)
print("DATASET READY FOR DASHBOARD REPORTING!")
print("=" * 60)

DATA CLEANING & FEATURE ENGINEERING SUMMARY

✓ Original Dataset: 12,575 rows
✓ Cleaned Dataset: 11,362 rows (90.4% retained)
✓ Total Features: 28
✓ Missing Values: 0

FEATURE CATEGORIES

1. Date Features:
   - Year, Month, Month_Name, Quarter, Day, Day_of_Week, Week_of_Year

2. Product Features:
   - Item_Number (extracted from Item column)

3. Revenue & Discount Features:
   - Revenue_Without_Discount, Discount_Amount, Discount_Percentage
   - Revenue_Segment, Quantity_Segment

4. Customer RFM Features:
   - Transaction_Count, Total_Customer_Value, Recency_Days
   - Customer_Segment (Bronze/Silver/Gold/Platinum)

DATASET READY FOR DASHBOARD REPORTING!


---

### I created an ETL Pipeline on Databricks for Data Cleaning and Reporting Dashboard. 

Link to the Dashboard: [Click Here](https://dbc-4deb77d6-9b56.cloud.databricks.com/dashboardsv3/01f16304980f1152be26ccce2f3813a4/published?o=7474658883336042)
